# 기계학습 기반 의미역 분석 (Machine Learning-based SRL)

- 학습 데이터: PropBank/FrameNet처럼 “동사 + 논항 + 역할 라벨” 형태를 갖춤(수업용 소규모 예제)

- 특징(Feature)

  - 구문 트리/의존관계: CAND_DEP, 술어–후보 간 의존거리(DEP_PATH_LEN)

  - 품사: CAND_POS, PRED_POS

  - 주변 단어: CTX_LEFT, CTX_RIGHT, HEAD_LEMMA

  - 전치사 단서: HAS_PREP_TO(수혜자/수취자 단서)

- 모델: LogisticRegression으로 Arg0/Arg1/Arg2/O 분류

- 출력: 정확도/분류리포트 + 새 문장에 대한 역할 예측(해석 가능한 피처 기반)

spacy: 문장을 구조적으로 분석해주는 NLP 툴킷

DictVectorizer: 문자 기반 특징 → 숫자 벡터 변환

LogisticRegression: 역할 분류기 (Arg0/Arg1/Arg2 등)

train_test_split, classification_report: 학습/평가 기본 절차

In [43]:
# ============================================
# SRL (Machine Learning-based) Mini Demo
# - 특징(Feature): 구문 트리(의존관계), 품사, 주변 단어
# - 분류기(Classifier): Logistic Regression
# ============================================

# spaCy: 영어 문장을 분석(토큰화, 품사 태깅, 의존 구문 분석 등)할 수 있는 라이브러리
import spacy

# DictVectorizer: 파이썬 딕셔너리 형태의 특징(feature)을
# 머신러닝 모델이 사용할 수 있는 숫자 벡터로 변환해주는 도구
from sklearn.feature_extraction import DictVectorizer

# LogisticRegression: 간단하고 널리 쓰이는 지도학습 분류기
from sklearn.linear_model import LogisticRegression

# train_test_split: 데이터를 학습용과 평가용으로 나누는 함수
from sklearn.model_selection import train_test_split

# classification_report, accuracy_score: 모델 성능 평가를 위한 지표
from sklearn.metrics import classification_report, accuracy_score

# 1) spaCy 영어 모델 로드
#    "en_core_web_sm" = 작은 영어 모델 (POS 태깅, 의존관계, 개체명 인식 가능)
#    만약 설치가 안 되어 있으면 아래 명령어로 설치해야 함:
#    !python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

아래 데이터는 PropBank/FrameNet 태깅 데이터의 축소판

핵심 역할(Arg0, Arg1, Arg2) + 부가 요소(O)를 손으로 붙여놓음

이후 이 데이터를 학습용 feature 추출 → 분류기 훈련 → 새 문장 예측에 사용

In [45]:
# ----------------------------------------------------
# 2) 수업용 소규모 데이터셋
# ----------------------------------------------------
# 실제 연구에서는 PropBank / FrameNet 같은 의미역이 태깅된 대규모 코퍼스를 사용해야 함
# 하지만 수업에서는 개념 이해를 위해 "작은 예제 데이터셋"을 직접 만듦
#
# - 각 항목은 하나의 문장(sent)과
#   술어 동사(pred_lemma), 그리고 후보 논항(candidates)을 포함
#
# - candidates: (단어/구, 의미역 라벨) 쌍
#   Arg0 = 주체(Agent, Seller, Giver)
#   Arg1 = 대상(Theme, Goods)
#   Arg2 = 수혜자/수취자(Recipient, Buyer)
#   O    = 기타(시간, 장소 등 핵심 역할이 아닌 성분)
# ----------------------------------------------------
data = [
    {
        "sent": "Mary gave John a book.",   # 문장
        "pred_lemma": "give",               # 술어 동사(원형)
        "candidates": [("Mary", "Arg0"),    # Mary = 주체 (Agent)
                       ("John", "Arg2"),    # John = 수혜자 (Recipient)
                       ("a book", "Arg1")]  # a book = 대상 (Theme)
    },
    {
        "sent": "John sent a letter to Mary.",
        "pred_lemma": "send",
        "candidates": [("John", "Arg0"),       # John = 주체 (Sender)
                       ("a letter", "Arg1"),   # a letter = 대상 (Theme)
                       ("Mary", "Arg2")]       # Mary = 수취자 (Recipient)
    },
    {
        "sent": "He sold the car to Mary.",
        "pred_lemma": "sell",
        "candidates": [("He", "Arg0"),         # He = 주체 (Seller)
                       ("the car", "Arg1"),    # the car = 대상 (Goods)
                       ("Mary", "Arg2")]       # Mary = 구매자 (Buyer)
    },
    {
        "sent": "They shipped the package to the customer yesterday.",
        "pred_lemma": "ship",
        "candidates": [("They", "Arg0"),             # 주체 (Shipper)
                       ("the package", "Arg1"),      # 대상 (Goods)
                       ("the customer", "Arg2"),     # 수취자 (Recipient)
                       ("yesterday", "O")]           # 시간 (기타)
    },
    {
        "sent": "Susan mailed a postcard to Tom on Monday.",
        "pred_lemma": "mail",
        "candidates": [("Susan", "Arg0"),            # 주체 (Sender)
                       ("a postcard", "Arg1"),       # 대상 (Theme)
                       ("Tom", "Arg2"),              # 수취자 (Recipient)
                       ("Monday", "O")]              # 시간 (기타)
    },
]

### 유틸 함수 설명

span_to_token: 문자열 스팬을 토큰(head)로 매핑 → 모델 입력 특징의 기준점 확보

find_predicate_token: 술어 동사 토큰을 안정적으로 찾기

dep_distance: 술어↔후보 구문적 거리를 수치화 → 역할과의 관련성 힌트

extract_features: 품사/의존/문맥/전치사 단서를 한데 모아 지도학습 입력 준비

In [46]:
# ----------------------------------------------------
# 3) 유틸: (문장, 술어, 후보 스팬) → 특징 사전 추출
#    - POS: 후보의 품사
#    - DEP: 후보의 의존관계 레이블
#    - PRED_POS: 술어 품사
#    - DEP_PATH_LEN: 술어와 후보 사이 의존경로 길이(트리 거리)
#    - HEAD/CTX: 후보의 head lemma, 좌/우 주변 단어(윈도우 1)
# ----------------------------------------------------

def span_to_token(doc, span_text):
    """문장(doc)에서 문자열 span_text가 가리키는 구간을 찾아
    그 구간의 대표 토큰(root/head)을 반환."""
    # 문장 전체 문자열에서 해당 스팬의 시작 인덱스 검색
    start = doc.text.find(span_text)
    if start < 0:                    # 못 찾으면 None
        return None
    # 스팬의 끝 인덱스 계산
    end = start + len(span_text)
    # 문자 위치를 토큰 스팬으로 매핑(어긋나면 주변까지 확장해 맞춤)
    span = doc.char_span(start, end, alignment_mode="expand")
    if span is None:                 # 매핑 실패 시 None
        return None
    return span.root                 # 스팬의 대표 토큰(헤드)을 사용

def find_predicate_token(doc, pred_lemma):
    """문장 안에서 술어 동사의 lemma가 pred_lemma인 토큰을 찾음.
    우선 VERB 품사에서 찾고, 없으면 lemma 일치 토큰을 폭넓게 탐색."""
    for tok in doc:
        if tok.pos_ == "VERB" and tok.lemma_ == pred_lemma:
            return tok               # 동사인 경우를 최우선 반환
    for tok in doc:
        if tok.lemma_ == pred_lemma:
            return tok               # 동사가 아니어도 lemma가 같으면 보조적으로 반환
    return None                      # 없으면 None

def dep_distance(tok_a, tok_b):
    """의존 구문 트리에서 두 토큰 사이의 거리(경로 길이)를 근사 계산."""
    # 각 토큰에서 조상 방향으로 올라가며 (토큰→root) 깊이를 기록
    a_anc = {a: i for i, a in enumerate([tok_a] + list(tok_a.ancestors))}
    b_anc = {b: i for i, b in enumerate([tok_b] + list(tok_b.ancestors))}
    # 공통 조상(least common ancestor 후보) 집합
    inter = set(a_anc.keys()) & set(b_anc.keys())
    if not inter:                    # 트리가 끊어졌다면 큰 값으로 처리
        return 999
    # 공통 조상까지의 단계 수 합의 최소값을 거리로 사용
    return min(a_anc[x] + b_anc[x] for x in inter)

def extract_features(doc, pred_tok, cand_tok):
    """후보 토큰(cand_tok)에 대해 SRL 분류에 쓸 특징들을 딕셔너리로 생성."""
    # 후보 토큰 좌/우 한 칸의 lemma (문장 경계면 특수 토큰 사용)
    left = doc[cand_tok.i - 1].lemma_ if cand_tok.i - 1 >= 0 else "<BOS>"
    right = doc[cand_tok.i + 1].lemma_ if cand_tok.i + 1 < len(doc) else "<EOS>"

    # 후보 토큰의 head(지배어) lemma (없으면 <NONE>)
    head_lemma = cand_tok.head.lemma_ if cand_tok.head is not None else "<NONE>"

    # 술어-후보 사이 의존 트리 거리
    dist = dep_distance(pred_tok, cand_tok)

    # 다양한 범주의 특징들을 하나의 dict로 정리
    feats = {
        "CAND_TEXT": cand_tok.text.lower(),      # 후보 원문(소문자)
        "CAND_LEMMA": cand_tok.lemma_.lower(),   # 후보 표제어(lemma)
        "CAND_POS": cand_tok.pos_,               # 후보 품사(POS)
        "CAND_DEP": cand_tok.dep_,               # 후보의 의존관계 레이블
        "HEAD_LEMMA": head_lemma,                # 후보의 head lemma
        "PRED_LEMMA": pred_tok.lemma_.lower(),   # 술어 lemma
        "PRED_POS": pred_tok.pos_,               # 술어 품사
        "DEP_PATH_LEN": dist,                    # 의존 경로 길이(가까울수록 관련성↑ 가설)
        "CTX_LEFT": left,                        # 좌측 한 단어 단서
        "CTX_RIGHT": right,                      # 우측 한 단어 단서
        # 전치사 'to' 단서(수혜자/수취자에 자주 등장) 존재 여부 (있으면 1, 없으면 0)
        "HAS_PREP_TO": int(any(tok.lower_ == "to" for tok in cand_tok.subtree)) \
                       or int(cand_tok.dep_.upper() == "POBJ" and cand_tok.head.lower_ == "to"),
    }
    return feats

data: 원본 코퍼스(문장+술어+후보+라벨)

X_dict: 각 후보별 특징 딕셔너리 모음 (머신러닝 입력용)

y: 각 후보의 정답 라벨 (머신러닝 출력용)

이 과정을 거치면 → 지도학습 데이터셋 (X, y) 완성

In [47]:
# ----------------------------------------------------
# 4) 데이터 전개:
# 각 문장에서 후보 스팬을 토큰으로 매핑하고
# (문장, 술어, 후보) → 특징(feats), 라벨(label) 쌍을 생성
# ----------------------------------------------------

# 특징 사전들을 담을 리스트, 정답 라벨들을 담을 리스트
X_dict, y = [], []

# 준비된 소규모 데이터셋(data)을 한 문장씩 처리
for ex in data:
    # spaCy로 문장 분석 (토큰화, 품사 태깅, 의존 분석 등)
    doc = nlp(ex["sent"])

    # 문장에서 술어 동사 토큰 찾기 (예: "give", "send", "sell" 등)
    pred_tok = find_predicate_token(doc, ex["pred_lemma"])
    if pred_tok is None:   # 술어를 못 찾으면 건너뜀
        continue

    # 후보 논항(candidates)에 대해 반복
    for span_text, label in ex["candidates"]:
        # 후보 문자열("Mary", "a book" 등)을 spaCy 토큰으로 매핑
        cand_tok = span_to_token(doc, span_text)
        if cand_tok is None:   # 매핑 실패 시 건너뜀
            continue

        # (문장, 술어, 후보) → 특징 추출
        feats = extract_features(doc, pred_tok, cand_tok)

        # 특징 사전 리스트에 추가
        X_dict.append(feats)

        # 정답 라벨(Arg0, Arg1, Arg2, O)을 y 리스트에 추가
        y.append(label)

# 총 학습 샘플 수와 라벨 종류를 출력
print(f"Total training samples: {len(y)}  | Labels: {sorted(set(y))}")

Total training samples: 17  | Labels: ['Arg0', 'Arg1', 'Arg2', 'O']


DictVectorizer가 문자형 특징을 숫자화해 분류기에 투입 가능하게 한다.

train_test_split(..., stratify=y)로 라벨 비율 유지 → 공정한 평가

classification_report로 Arg0/Arg1/Arg2/O별 성능을 확인해 어떤 역할이 잘/덜 맞춰졌는지 파악할 수 있다.

In [49]:
# ----------------------------------------------------
# 5) 벡터화(딕셔너리 → 숫자 벡터) + 학습/평가
# ----------------------------------------------------

# DictVectorizer:
#  - {'CAND_POS':'NOUN', 'DEP_PATH_LEN':2, ...} 같은 딕셔너리를
#    머신러닝이 읽을 수 있는 희소 벡터(원-핫/수치)로 바꿔줌
vec = DictVectorizer(sparse=True)

# X_dict(특징 딕셔너리 리스트)를 실제 숫자 행렬 X로 변환
X = vec.fit_transform(X_dict)   # fit: 특징 사전 학습 / transform: 숫자화

# 학습/평가 데이터 분할 (학습 70% / 평가 30%)
# stratify=y → 라벨 분포를 학습/평가에 동일하게 유지(클래스 불균형 완화)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 분류기 준비: 로지스틱 회귀(다중 클래스 자동 처리)
clf = LogisticRegression(max_iter=1000, multi_class="auto")

# 모델 학습(훈련 데이터로 계수/가중치 학습)
clf.fit(X_tr, y_tr)

# 평가: 테스트 데이터 예측
pred = clf.predict(X_te)

# 정확도와 상세 리포트 출력
print("\n[Evaluation]")
print("Accuracy:", round(accuracy_score(y_te, pred), 3))
# 클래스별 precision/recall/F1, 지원수(support) 등을 확인
print(classification_report(y_te, pred, zero_division=0))


[Evaluation]
Accuracy: 1.0
              precision    recall  f1-score   support

        Arg0       1.00      1.00      1.00         2
        Arg1       1.00      1.00      1.00         2
        Arg2       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


아래 함수는 새로운 문장을 입력받아, 학습된 SRL 분류기로 후보 논항에 Arg0/Arg1/Arg2/O 라벨을 자동 부여한다.

In [50]:
# ----------------------------------------------------
# 6) 새 문장 테스트: 모델이 논항에 어떤 의미역을 붙이는지 확인
# ----------------------------------------------------
def predict_roles(sentence, pred_lemma, spans):
    """
    sentence   : 새 입력 문장 (예: "Mary gave John a book.")
    pred_lemma : 술어 동사의 원형 (예: 'give', 'sell')
    spans      : 의미역을 예측해볼 후보 구(문자열) 리스트
                 (예: ["Mary", "John", "a book"])
    """

    # 문장 분석 (spaCy로 토큰화, 품사, 의존관계 분석 등)
    doc = nlp(sentence)

    # 문장에서 술어 동사 토큰 찾기
    pred_tok = find_predicate_token(doc, pred_lemma)
    if pred_tok is None:
        print("(!) 술어(predicate)를 찾지 못했습니다.")
        return

    # 문장과 술어 출력
    print(f"\n[Sentence] {sentence}")
    print(f"  Predicate: {pred_tok.text} (lemma={pred_tok.lemma_})")

    # 각 후보 span(문자열)에 대해 반복
    for st in spans:
        # 후보 문자열을 토큰 객체로 매핑
        cand_tok = span_to_token(doc, st)
        if cand_tok is None:
            # 문장에서 매칭 실패 시 메시지 출력
            print(f"  - {st:<12} → (span not found)")
            continue

        # 후보에 대한 특징(feature) 추출
        feats = extract_features(doc, pred_tok, cand_tok)

        # 특징을 벡터화하여 분류기에 입력
        x = vec.transform([feats])

        # 학습된 모델로 의미역 라벨 예측
        label = clf.predict(x)[0]

        # 결과 출력 (예: Arg0, Arg1, Arg2, O)
        print(f"  - {st:<12} → predicted role: {label}")

predict_roles 함수가 새로운 문장을 분석해서 각 후보 span에 의미역(Arg0/Arg1/Arg2/O)을 예측해준다.

In [51]:
# 데모 1
predict_roles(
    "Mary gave John a book in the morning.",
    pred_lemma="give",
    spans=["Mary", "John", "a book", "the morning"]
)


[Sentence] Mary gave John a book in the morning.
  Predicate: gave (lemma=give)
  - Mary         → predicted role: Arg0
  - John         → predicted role: Arg2
  - a book       → predicted role: Arg1
  - the morning  → predicted role: Arg2


In [52]:
# 데모 2
predict_roles(
    "He sold the car to Mary yesterday.",
    pred_lemma="sell",
    spans=["He", "the car", "Mary", "yesterday"]
)


[Sentence] He sold the car to Mary yesterday.
  Predicate: sold (lemma=sell)
  - He           → predicted role: Arg0
  - the car      → predicted role: Arg1
  - Mary         → predicted role: Arg2
  - yesterday    → predicted role: O
